### Importing Libraries

In [ ]:
# Import Libraries
import pandas as pd
import praw
from datetime import datetime
import os
from dotenv import load_dotenv

load_dotenv()

### Accessing the Reddit API

In [ ]:
user_agent    = os.getenv("REDDIT_USER_AGENT")
client_id     = os.getenv("REDDIT_CLIENT_ID")
client_secret = os.getenv("REDDIT_CLIENT_SECRET")

reddit = praw.Reddit(
    client_id=client_id,
    client_secret=client_secret,
    user_agent=user_agent
)

print("Connected to Reddit API successfully!")

### Configuration

Set the target subreddit and the keywords used to **tag** posts.
All posts are collected — keywords are recorded as metadata for later filtering.

In [ ]:
SUBREDDIT = ""   # Change to target a different subreddit

key_words = []

OUTPUT_FILE = f"AllPost_{SUBREDDIT}.xlsx"

print(f"Subreddit : r/{SUBREDDIT}")
print(f"Keywords  : {len(key_words)} terms")
print(f"Output    : {OUTPUT_FILE}")

### Scraping Posts

In [ ]:
collected_posts = []
collection_timestamp = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")

for submission in reddit.subreddit(SUBREDDIT).top(time_filter="all", limit=None):
    
    if submission.is_self:
        post_type = "text"
    elif submission.url.endswith((".jpg", ".jpeg", ".png", ".gif", ".webp")) \
         or "i.redd.it" in submission.url or "imgur.com" in submission.url:
        post_type = "image"
    else:
        post_type = "link"

    combined_text = (submission.title + " " + submission.selftext).lower()
    matched = [kw for kw in key_words if kw.lower() in combined_text]

    collected_posts.append({
        "Title"            : submission.title,
        "Author"           : str(submission.author) if submission.author else None,
        "Upvotes"          : submission.score,
        "URL"              : submission.url,
        "Text"             : submission.selftext,
        "Post_Type"        : post_type,
        "Matched_Keywords" : ", ".join(matched) if matched else None,
        "Collected_At"     : collection_timestamp,
    })

print(f"Total posts collected : {len(collected_posts)}")
print(f"Collected at          : {collection_timestamp}")

### Organizing into a DataFrame

In [ ]:
df = pd.DataFrame(collected_posts)
print(df)

In [ ]:
df.info()

### Summary by Post Type and Keyword Matches

In [ ]:
print(df["Post_Type"].value_counts())
print()
print(f"Posts with keyword match : {df['Matched_Keywords'].notna().sum()}")
print(f"Posts without match      : {df['Matched_Keywords'].isna().sum()}")
print(f"Null authors             : {df['Author'].isna().sum()}")

### First 5 rows

In [ ]:
df.head()

### Export to Excel

In [ ]:
df.to_excel(OUTPUT_FILE, index=False)
print(f"Saved {len(df)} posts from r/{SUBREDDIT} to '{OUTPUT_FILE}'")